# 🏥 حل مشروع التشخيص الطبي الذكي والتعلم العميق (Healthcare Diagnostic Solution)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('datasets/health_risk_data.csv')
print(df.info())

### 📊 الجزء الأول: تنظيف البيانات والقيم المفقودة

In [ ]:
# Fill numeric missing values
for col in ['bmi', 'cholesterol', 'exercise_hours_per_week', 'blood_sugar']:
    df[col] = df[col].fillna(df[col].median())

# Standardize gender
df['gender'] = df['gender'].replace({'M': 'Male', 'male': 'Male', 'F': 'Female', 'female': 'Female'})
df['smoking_status'] = df['smoking_status'].replace({'never': 'Never', 'current': 'Current', 'Yes': 'Current', 'No': 'Never'})
print(df['gender'].value_counts())

### 🤖 الجزء الثاني: مقارنة نماذج الذكاء الاصطناعي ورسم ROC Curve

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import roc_curve, auc, accuracy_score

# Categorical encoding
X = df.drop(columns=['patient_id', 'risk_score', 'heart_disease'])
y = df['heart_disease']
X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    "Logistic Regression": LogisticRegression(probability=True if hasattr(LogisticRegression, 'predict_proba') else False),
    "Random Forest": RandomForestClassifier(random_state=42),
    "SVM": SVC(probability=True, random_state=42)
}

plt.figure(figsize=(10, 8))
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    probs = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], 'k--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Comparison')
plt.legend(loc="lower right")
plt.savefig('solutions/health_roc_comparison.png')
plt.show()

### 🧠 الجزء الثالث: التعلم العميق (Deep Learning)

In [ ]:
# 1. Forward Pass from scratch
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def forward_neuron(inputs, weights, bias):
    return sigmoid(np.dot(inputs, weights) + bias)

inputs = np.array([25.4, 130.0, 210.0]) # BMI, BP, Chol
weights = np.array([0.05, 0.02, 0.01])
bias = -5.0
print("Single Neuron Prediction Output:", forward_neuron(inputs, weights, bias))

# 2. MLP Classifier in action
from sklearn.neural_network import MLPClassifier
mlp = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300, random_state=42)
mlp.fit(X_train_scaled, y_train)
mlp_preds = mlp.predict(X_test_scaled)
print(f"MLP Neural Network Accuracy: {accuracy_score(y_test, mlp_preds):.4f}")